In [1]:
import os
import getpass

# If you haven't set your API key, this will prompt you to enter it.
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key:")

In [2]:
# Install required packages
!pip install -q \
    langchain \
    langchain-community \
    langchain-openai \
    langchain-experimental \
    langchain-chroma \
    langchain-text-splitters \
    tiktoken \
    chromadb \
    rank_bm25 \
    numpy

## Chunking Strategies

**Evaluating the ideal chunk size**

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# A sample document with a clear narrative structure
document_text = """
The human heart is a muscular organ that pumps blood through the circulatory system.
This blood delivers oxygen and nutrients to the body and removes metabolic waste. The heart
has four chambers: the right atrium, the right ventricle, the left atrium, and the left ventricle.
The average adult heart beats about 60 to 100 times per minute. The circulatory system consists of the heart,
blood vessels, and blood. It is a vital part of the human body, responsible for transporting essential substances.
A healthy lifestyle, including regular exercise and a balanced diet, is crucial for cardiovascular health.
"""

# Save the text to a file
with open("heart_document.txt", "w") as f:
    f.write(document_text)

# Load the document
loader = TextLoader("heart_document.txt")
documents = loader.load()

# Define chunk sizes for the experiment
chunk_sizes = [50, 150, 400]  # Small, Medium, Large

# A dictionary to hold our chunks for each size
chunked_documents = {}

for size in chunk_sizes:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=int(size * 0.1)  # 10% overlap
    )
    chunks = text_splitter.split_documents(documents)
    chunked_documents[size] = chunks
    print(f"Split document into {len(chunks)} chunks of size {size}.")

Split document into 16 chunks of size 50.
Split document into 6 chunks of size 150.
Split document into 2 chunks of size 400.


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Initialize the OpenAI embeddings model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# A dictionary to hold our ChromaDB vector stores
vector_dbs = {}

for size, chunks in chunked_documents.items():
    # Create a persistent ChromaDB instance for each chunk size
    db_path = f"./chroma_db_size_{size}"
    db = Chroma.from_documents(
        chunks,
        embeddings,
        persist_directory=db_path
    )
    vector_dbs[size] = db
    print(f"Created ChromaDB instance for chunk size {size}.")

In [5]:
# A complex query that benefits from a broader context
query = "What is the primary function of the heart and its connection to the circulatory system?"

# The number of documents to retrieve
k = 3

print("\n" + "="*50)
print(f"Querying with: '{query}'")
print("="*50 + "\n")

for size, db in vector_dbs.items():
    print("-" * 30)
    print(f"Retrieving with chunk size: {size}")

    # Perform a similarity search
    retrieved_docs = db.similarity_search(query, k=k)

    for i, doc in enumerate(retrieved_docs):
        print(f"  Result {i+1}:")
        print(f"  {doc.page_content}\n")


Querying with: 'What is the primary function of the heart and its connection to the circulatory system?'

------------------------------
Retrieving with chunk size: 50
  Result 1:
  the body and removes metabolic waste. The heart

  Result 2:
  The human heart is a muscular organ that pumps

  Result 3:
  blood through the circulatory system.

------------------------------
Retrieving with chunk size: 150
  Result 1:


KeyboardInterrupt: 

**Fixed Length Chunking**

In [1]:
from langchain_text_splitters import CharacterTextSplitter

# Example text
text = """
    # Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables computers to learn and improve from experience without being explicitly programmed.

    ## Types of Machine Learning

    ### Supervised Learning
    Supervised learning uses labeled training data to learn a mapping function from input to output. Common examples include:
    - Classification: Predicting categories (spam/not spam)
    - Regression: Predicting continuous values (house prices)

    ### Unsupervised Learning
    Unsupervised learning finds hidden patterns in data without labeled examples:
    - Clustering: Grouping similar data points
    - Dimensionality reduction: Reducing feature space

    ### Reinforcement Learning
    An agent learns through interaction with an environment, receiving rewards or penalties for actions taken.
    """

# Initialize the CharacterTextSplitter with a fixed chunk size and no overlap
fixed_length_splitter = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    separator="\n"
)

# Split the text
chunks = fixed_length_splitter.split_text(text)

# Print the resulting chunks
print("Chunks using fixed-length splitter:")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} (length: {len(chunk)}):")
    print(chunk)
    print("-" * 20)

Chunks using fixed-length splitter:
Chunk 1 (length: 188):
# Machine Learning Fundamentals
    Machine learning is a subset of artificial intelligence that enables computers to learn and improve from experience without being explicitly programmed.
--------------------
Chunk 2 (length: 182):
## Types of Machine Learning
    ### Supervised Learning
    Supervised learning uses labeled training data to learn a mapping function from input to output. Common examples include:
--------------------
Chunk 3 (length: 147):
- Classification: Predicting categories (spam/not spam)
    - Regression: Predicting continuous values (house prices)
    ### Unsupervised Learning
--------------------
Chunk 4 (length: 179):
Unsupervised learning finds hidden patterns in data without labeled examples:
    - Clustering: Grouping similar data points
    - Dimensionality reduction: Reducing feature space
--------------------
Chunk 5 (length: 137):
### Reinforcement Learning
    An agent learns through interactio

**Semantic Chunking**

In [2]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings

text_splitter = SemanticChunker(OpenAIEmbeddings())
docs = text_splitter.create_documents([text])

for i, doc in enumerate(docs):
    print(f"Chunk {i+1}:")
    print(doc.page_content)
    print("-" * 50)

Chunk 1:

    # Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables computers to learn and improve from experience without being explicitly programmed. ## Types of Machine Learning

    ### Supervised Learning
    Supervised learning uses labeled training data to learn a mapping function from input to output.
--------------------------------------------------
Chunk 2:
Common examples include:
    - Classification: Predicting categories (spam/not spam)
    - Regression: Predicting continuous values (house prices)

    ### Unsupervised Learning
    Unsupervised learning finds hidden patterns in data without labeled examples:
    - Clustering: Grouping similar data points
    - Dimensionality reduction: Reducing feature space

    ### Reinforcement Learning
    An agent learns through interaction with an environment, receiving rewards or penalties for actions taken. 
--------------------------------------------------


**Document structure based chunking**

In [4]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

md_header_splits = markdown_splitter.split_text(text)

print("Chunks using Markdown Header Splitter:")
for i, doc in enumerate(md_header_splits):
    print(f"Chunk {i+1}:")
    print(doc.page_content)
    print(f"Metadata: {doc.metadata}")
    print("-" * 50)

Chunks using Markdown Header Splitter:
Chunk 1:
Machine learning is a subset of artificial intelligence that enables computers to learn and improve from experience without being explicitly programmed.
Metadata: {'Header 1': 'Machine Learning Fundamentals'}
--------------------------------------------------
Chunk 2:
Supervised learning uses labeled training data to learn a mapping function from input to output. Common examples include:
- Classification: Predicting categories (spam/not spam)
- Regression: Predicting continuous values (house prices)
Metadata: {'Header 1': 'Machine Learning Fundamentals', 'Header 2': 'Types of Machine Learning', 'Header 3': 'Supervised Learning'}
--------------------------------------------------
Chunk 3:
Unsupervised learning finds hidden patterns in data without labeled examples:
- Clustering: Grouping similar data points
- Dimensionality reduction: Reducing feature space
Metadata: {'Header 1': 'Machine Learning Fundamentals', 'Header 2': 'Types of Machi

**Hypothetical Document Embedding**

In [6]:
import numpy as np
from rank_bm25 import BM25Okapi

# Example query and product titles
query = "I want running shoes"
documents = [
    "Adidas Ultraboost sneakers for men , Marathon Shoes",
    "Nike Air Zoom Pegasus trainers",
    "Puma casual footwear for everyday wear",
    "Woodland leather trekking boots",
    "Asics Gel-Kayano stability trainers",
    "Reebok CrossFit gym footwear",
    "Clarks formal leather boots",
    "Skechers lightweight walking sneakers",
    "New Balance Fresh Foam trail trainers",
    "Bata budget-friendly casual footwear",
    "Fila sports footwear for gym workouts",
    "Converse high-top casual sneakers"
]

# Tokenize documents
tokenized_docs = [doc.lower().split() for doc in documents]

# Initialize BM25
bm25 = BM25Okapi(tokenized_docs)

hyde_answer = """Nike Air Jordan – Lightweight daily trainer with responsive cushioning.

Adidas Ultraboost 23 – High comfort running shoe with energy-returning midsole.

Asics Gel-Kayano 30 – Stability shoe designed for overpronators and long runs.

New Balance Fresh Foam 1080v13 – Plush cushioned shoe for marathon training."""

# Retrieval without HyDE
tokenized_query = query.lower().split()
sims_no_hyde = bm25.get_scores(tokenized_query)

# Filter out only positive matches
threshold = 0.1
top_docs_no_hyde = [i for i in np.argsort(sims_no_hyde)[::-1] if sims_no_hyde[i] > threshold][:3]

print("Top product titles WITHOUT HyDE (BM25):")
if not top_docs_no_hyde:
    print("- No relevant results found")
else:
    for idx in top_docs_no_hyde:
        print(f"- {documents[idx]} (score: {sims_no_hyde[idx]:.2f})")

# Retrieval with HyDE (query + hypothetical answer)
hyde_query = query + " " + hyde_answer
tokenized_hyde_query = hyde_query.lower().split()
sims_hyde = bm25.get_scores(tokenized_hyde_query)
top_docs_hyde = np.argsort(sims_hyde)[::-1][:3]

print("\nTop product titles WITH HyDE (BM25):")
for idx in top_docs_hyde:
    print(f"- {documents[idx]} (score: {sims_hyde[idx]:.2f})")

Top product titles WITHOUT HyDE (BM25):
- Adidas Ultraboost sneakers for men , Marathon Shoes (score: 1.59)

Top product titles WITH HyDE (BM25):
- Adidas Ultraboost sneakers for men , Marathon Shoes (score: 7.91)
- New Balance Fresh Foam trail trainers (score: 7.41)
- Asics Gel-Kayano stability trainers (score: 6.67)
